In [ ]:
# libraries
# install once if needed: %pip install torch numpy pandas seaborn matplotlib pillow nbformat IPython scipy
import os
from pathlib import Path
import nbformat
import pandas as pd
from IPython.display import display

In [ ]:
# globals; local smoke and full VACC runs have separate named output directories
ROOT = Path.cwd().resolve()
PROFILE = 'smoke'  # smoke / pilot / full / benchmark
DEVICE = 'auto'  # local: auto / cpu / cuda; Slurm always requests CUDA
RESUME = False
EXPERIMENT = 'smoke-local-001'  # use a new name for a new experiment
MODE = 'local'  # local / slurm
SLURM = {'account': '', 'max_parallel': 2, 'walltime': '24:00:00', 'memory': '32G', 'cpus': 2}
DRY_RUN = True  # Slurm: inspect commands first, then False to submit
ALLOW_DEVICE_CHANGE = False  # v2 checkpoints only; logs a new CUDA RNG stream on migration
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [ ]:
# orchestration functions; definitions only
tools_nb = nbformat.read(ROOT / 'pipeline_tools.ipynb', 4)
for code in tools_nb.cells:
    exec(compile(code.source, 'pipeline_tools.ipynb', 'exec'), globals())

In [ ]:
# one entry point; Slurm runs the same scientific notebooks and preserves their PDP links
if MODE == 'local':
    result_root = run_local(ROOT, EXPERIMENT, PROFILE, DEVICE, RESUME)
    display(status(ROOT, EXPERIMENT))
    if PROFILE != 'benchmark':
        display(pd.read_csv(result_root / 'h_compare/output/comparison.csv'))
elif MODE == 'slurm':
    submission = submit_slurm(ROOT, EXPERIMENT, PROFILE, resume=RESUME,
                              allow_device_change=ALLOW_DEVICE_CHANGE, dry_run=DRY_RUN, **SLURM)
    display(pd.DataFrame(submission['submitted']))
else:
    raise ValueError('MODE must be local or slurm')